# Titanic Survival Prediction - Data Preprocessing

## Objective

The purpose of this notebook is to prepare the Titanic dataset for machine learning model development.

Based on the findings from the Exploratory Data Analysis (EDA), the following preprocessing tasks will be performed:

- Remove unnecessary features
- Handle missing values
- Create engineered features
- Encode categorical variables
- Scale numerical variables
- Prepare the final dataset for model training

The final output of this notebook will be a clean, model-ready dataset.


In [5]:
#import all the related library:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

from pathlib import Path

## Load Dataset

Load the raw Titanic dataset before applying any preprocessing transformations.

In [8]:
df = pd.read_csv("data/titanic.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Create Working Copy

A copy of the original dataset is created to ensure the raw dataset remains unchanged.

In [9]:
df_processed = df.copy()
print("Data Shape:", df_processed.shape)

Data Shape: (891, 12)


## Remove Unnecessary Features

Based on the EDA findings:

- PassengerId is a unique identifier.
- Ticket contains high-cardinality values.
- Cabin contains excessive missing values.

These features are removed because they are unlikely to contribute significantly to model performance.

In [11]:
columns_to_drop = [
    "PassengerId",
    "Ticket",
    "Cabin"
]

df_processed.drop(
    columns=columns_to_drop,
    inplace=True
)

df_processed.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S


## Handle Missing Values

The EDA identified missing values in:

- Age
- Embarked

The following imputation techniques will be applied:

| Feature | Strategy |
|----------|----------|
| Age | Median |
| Embarked | Most Frequent Value |

In [13]:
df_processed.isnull().sum()

Survived      0
Pclass        0
Name          0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

### Age Imputation

Missing age values are replaced using the median age because the median is less sensitive to extreme values and outliers.

In [14]:
median_age = df_processed["Age"].median()

df_processed["Age"] = (
    df_processed["Age"]
    .fillna(median_age)
)

### Embarked Imputation

Missing embarkation locations are replaced using the most frequently occurring category.

In [15]:
mode_embarked = (
    df_processed["Embarked"]
    .mode()[0]
)

df_processed["Embarked"] = (
    df_processed["Embarked"]
    .fillna(mode_embarked)
)

## Verify Missing Values

Verify that all missing values have been handled successfully.

In [16]:
df_processed.isnull().sum()

Survived    0
Pclass      0
Name        0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

## Feature Engineering

EDA revealed that family-related information may help predict passenger survival.

The following engineered features will be created:

### FamilySize

Represents the total number of family members travelling with a passenger.

Formula:

FamilySize = SibSp + Parch + 1

### IsAlone

Indicates whether the passenger travelled alone.

- 1 = Alone
- 0 = With Family

In [17]:
df_processed["FamilySize"] = (
    df_processed["SibSp"]
    + df_processed["Parch"]
    + 1
)

df_processed["IsAlone"] = np.where(
    df_processed["FamilySize"] == 1,
    1,
    0
)

df_processed[
    ["SibSp",
     "Parch",
     "FamilySize",
     "IsAlone"]
].head()

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


## Extract Passenger Title

Passenger names contain titles such as:

- Mr
- Mrs
- Miss
- Master

These titles may capture information related to age, social status, and family structure.

In [18]:
df_processed["Title"] = (
    df_processed["Name"]
    .str.extract(" ([A-Za-z]+)\.")
)

df_processed[
    ["Name", "Title"]
].head()

,Name,Title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,"Allen, Mr. William Henry",Mr


## Group Rare Titles

Rare titles appear infrequently and may introduce unnecessary complexity.

These titles will be grouped into a single category called "Rare".

In [19]:
df_processed["Title"] = (
    df_processed["Title"]
    .replace(
        [
            "Lady",
            "Countess",
            "Capt",
            "Col",
            "Don",
            "Dr",
            "Major",
            "Rev",
            "Sir",
            "Jonkheer",
            "Dona"
        ],
        "Rare"
    )
)

## Remove Original Name Feature

After extracting useful information from passenger names, the original Name column is no longer required.

In [20]:
df_processed.drop(
    columns=["Name"],
    inplace=True
)

## Save Cleaned Dataset

At this stage:

- Missing values have been handled
- Feature engineering has been completed
- Unnecessary features have been removed

Save this as the cleaned dataset before encoding and scaling.

In [22]:
df_processed.to_csv(
    "data/titanic_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


## Encode Categorical Variables

Machine learning algorithms require numerical inputs.

The following categorical features will be encoded:

- Sex
- Embarked
- Title

One-Hot Encoding is used to avoid introducing artificial ordering.

In [23]:
df_processed = pd.get_dummies(
    df_processed,
    columns=[
        "Sex",
        "Embarked",
        "Title"
    ],
    drop_first=True
)

df_processed.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rare
0,0,3,22.0,1,0,7.2500,2,0,1,0,1,0,0,0,1,0,0,0
1,1,1,38.0,1,0,71.2833,2,0,0,0,0,0,0,0,0,1,0,0
2,1,3,26.0,0,0,7.9250,1,1,0,0,1,1,0,0,0,0,0,0
3,1,1,35.0,1,0,53.1000,2,0,0,0,1,0,0,0,0,1,0,0
4,0,3,35.0,0,0,8.0500,1,1,1,0,1,0,0,0,1,0,0,0


## Separate Features and Target Variable

The target variable for this classification problem is:

Survived

- 0 = Did Not Survive
- 1 = Survived

In [24]:
X = df_processed.drop(
    columns=["Survived"]
)

y = df_processed["Survived"]

## Feature Scaling

Numerical features have different scales and units.

The following features will be standardized:

- Age
- Fare
- SibSp
- Parch
- FamilySize

In [25]:
scaler = StandardScaler()

numerical_columns = [
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "FamilySize"
]
X[numerical_columns]=scaler.fit_transform(
   X[numerical_columns])

## Save Model-Ready Dataset

The dataset is now fully prepared for machine learning and will be saved for model development.

In [26]:
model_ready_df = X.copy()

model_ready_df["Survived"] = y

model_ready_df.to_csv(
    "data/titanic_model_ready.csv",
    index=False
)

print("Model-ready dataset saved successfully.")

Model-ready dataset saved successfully.


# Preprocessing Summary

The following preprocessing steps were completed:

✅ Removed PassengerId, Ticket, and Cabin

✅ Imputed missing Age values using the median

✅ Imputed missing Embarked values using the mode

✅ Created FamilySize feature

✅ Created IsAlone feature

✅ Extracted passenger titles

✅ Encoded categorical variables

✅ Standardized numerical features

✅ Generated cleaned and model-ready datasets

The dataset is now ready for train-test splitting and model development.